# 🌳 Decision Tree — Applied on Titanic Dataset
**Topic:** End-to-End Classification with Decision Tree

---

## 📌 Is Notebook Mein Kya Karenge?

| Part | Task |
|------|------|
| A | Dataset load, EDA, preprocessing |
| B | Decision Tree baseline + depth analysis |
| C | Hyperparameter tuning (GridSearchCV) |
| D | Model interpretation + Error Analysis |

### 🚢 Dataset: Titanic (Seaborn)
- **Task:** Predict karo passenger survived (1) ya nahi (0)
- **Features:** age, sex, pclass, fare, sibsp, parch, embarked
- **Why Titanic?** Real-world data, missing values, mixed types — perfect for DT!

### 💡 Why Decision Tree for Titanic?
> Titanic survival kuch simple rules follow karta hai: **'Women and children first'**, first class passengers ko priority mili. Yeh rules ek Decision Tree naturally capture kar sakta hai!
>
> Actual rule: `sex=female → survived` / `sex=male AND pclass=1 → survived`
> Yahi ek tree automatically seekh jaata hai data se!

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder

# ===========================================
# Data Load
# ===========================================
df = sns.load_dataset('titanic')

print(f"Dataset Shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nTarget Distribution:")
print(df['survived'].value_counts())
print(f"\nSurvival Rate: {df['survived'].mean():.2%}")

---
## Part A: EDA + Data Preprocessing

### 🔍 Kya Dekhna Hai?
- Missing values kaun se features mein hain?
- Target class balance/imbalance?
- Kaunse features survival ke saath correlated hain?
- Categorical features ko kaise handle karein?

### 🎯 Yeh Step Kyun Zaroori Hai?
> Decision Tree missing values aur categorical data handle kar sakta hai, lekin sklearn implementation ke liye hume numeric input chahiye. EDA se samajh aata hai kaunse features informative hain — isse better model banate hain.

In [ ]:
# Missing values check
print("Missing Values:")
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Count': missing, 'Percentage': missing_pct})
print(missing_df[missing_df['Count'] > 0])

In [ ]:
# EDA: Key survival patterns
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

# 1. Survival by Sex
survival_sex = df.groupby('sex')['survived'].mean()
survival_sex.plot(kind='bar', ax=axes[0,0], color=['#e74c3c','#3498db'], rot=0)
axes[0,0].set_title('Survival Rate by Sex')
axes[0,0].set_ylabel('Survival Rate')
axes[0,0].set_ylim(0, 1)
for i, v in enumerate(survival_sex):
    axes[0,0].text(i, v+0.02, f'{v:.1%}', ha='center', fontweight='bold')

# 2. Survival by Pclass
survival_class = df.groupby('pclass')['survived'].mean()
survival_class.plot(kind='bar', ax=axes[0,1], color=['gold','silver','#cd7f32'], rot=0)
axes[0,1].set_title('Survival Rate by Passenger Class')
axes[0,1].set_xlabel('Pclass (1=First, 3=Third)')
axes[0,1].set_ylim(0, 1)
for i, v in enumerate(survival_class):
    axes[0,1].text(i, v+0.02, f'{v:.1%}', ha='center', fontweight='bold')

# 3. Age distribution by survival
df[df.survived==1]['age'].dropna().plot(kind='hist', ax=axes[0,2], alpha=0.6,
                                         label='Survived', color='#2ecc71', bins=20)
df[df.survived==0]['age'].dropna().plot(kind='hist', ax=axes[0,2], alpha=0.6,
                                         label='Died', color='#e74c3c', bins=20)
axes[0,2].set_title('Age Distribution by Survival')
axes[0,2].legend()

# 4. Fare vs Survival
df.boxplot(column='fare', by='survived', ax=axes[1,0])
axes[1,0].set_title('Fare Distribution by Survival')
axes[1,0].set_xlabel('Survived (0=No, 1=Yes)')

# 5. Pclass + Sex heatmap
pivot = df.pivot_table(values='survived', index='sex', columns='pclass', aggfunc='mean')
sns.heatmap(pivot, annot=True, fmt='.1%', cmap='RdYlGn', ax=axes[1,1])
axes[1,1].set_title('Survival Rate: Sex × Pclass')

# 6. Embarked survival
survival_emb = df.groupby('embarked')['survived'].mean().dropna()
survival_emb.plot(kind='bar', ax=axes[1,2], color='#9b59b6', rot=0, alpha=0.8)
axes[1,2].set_title('Survival Rate by Embarkation Port')
axes[1,2].set_ylim(0, 1)

plt.suptitle('Titanic EDA: Survival Patterns', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n📌 Key Insights:")
print("  1. Mahilaon ka survival rate bahut zyada hai (~74%)")
print("  2. 1st Class ka survival rate highest (~63%)")
print("  3. Children (age<10) zyada survive kiye")

In [ ]:
# ===========================================
# Preprocessing
# ===========================================

# Useful features select karo
features = ['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked']
df_model = df[features + ['survived']].copy()

# Missing values handle karo
df_model['age'].fillna(df_model['age'].median(), inplace=True)
df_model['embarked'].fillna(df_model['embarked'].mode()[0], inplace=True)
df_model['fare'].fillna(df_model['fare'].median(), inplace=True)

# Feature Engineering: Family size
df_model['family_size'] = df_model['sibsp'] + df_model['parch'] + 1
df_model['is_alone'] = (df_model['family_size'] == 1).astype(int)

# Categorical encode karo
df_model['sex_enc'] = (df_model['sex'] == 'female').astype(int)  # 1=female, 0=male
embarked_map = {'S': 0, 'C': 1, 'Q': 2}
df_model['embarked_enc'] = df_model['embarked'].map(embarked_map)

# Final features
feature_cols = ['pclass', 'sex_enc', 'age', 'fare', 'family_size', 'is_alone', 'embarked_enc']
X = df_model[feature_cols]
y = df_model['survived']

print(f"Final features: {feature_cols}")
print(f"Any missing values? {X.isnull().sum().sum()}")
print(f"\nX shape: {X.shape}, y shape: {y.shape}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {X_train.shape[0]} | Test: {X_test.shape[0]}")

---
## Part B: Decision Tree — Baseline + Depth Analysis

### 🎯 Is Part Mein Kya Sikhenge?
> Pehle ek **default tree** fit karke dekhenge — woh overfit hoga. Phir **depth vary karke** dekhenge kahan train-test gap minimize hota hai. Yahi **bias-variance tradeoff** hai trees mein.

**Rule of Thumb:**
- `max_depth` bahut kam → Underfitting (high bias)
- `max_depth` bahut zyada → Overfitting (high variance)
- Sweet spot → Cross-validation se dhundhte hain

In [ ]:
# ===========================================
# Baseline: Default Tree
# ===========================================
dt_base = DecisionTreeClassifier(random_state=42)
dt_base.fit(X_train, y_train)

print("=== Baseline Decision Tree ===")
print(f"Depth: {dt_base.get_depth()} | Leaves: {dt_base.get_n_leaves()}")
print(f"Train Accuracy: {dt_base.score(X_train, y_train):.4f}")
print(f"Test Accuracy:  {dt_base.score(X_test, y_test):.4f}")

# CV score
cv_base = cross_val_score(dt_base, X, y, cv=5)
print(f"CV Score (5-fold): {cv_base.mean():.4f} ± {cv_base.std():.4f}")

In [ ]:
# ===========================================
# Depth Analysis
# ===========================================
depths = range(1, 16)
train_acc, test_acc, cv_acc = [], [], []

for d in depths:
    dt = DecisionTreeClassifier(max_depth=d, random_state=42)
    dt.fit(X_train, y_train)
    train_acc.append(dt.score(X_train, y_train))
    test_acc.append(dt.score(X_test, y_test))
    cv_acc.append(cross_val_score(dt, X, y, cv=5).mean())

best_d = depths[np.argmax(cv_acc)]

plt.figure(figsize=(10, 5))
plt.plot(depths, train_acc, 'b-o', ms=5, label='Train')
plt.plot(depths, test_acc,  'r-s', ms=5, label='Test')
plt.plot(depths, cv_acc,    'g-^', ms=5, label='CV Mean')
plt.axvline(best_d, color='purple', linestyle='--', lw=2, label=f'Best depth={best_d}')
plt.fill_between(depths, train_acc, test_acc, alpha=0.1, color='red', label='Overfit Gap')
plt.xlabel('max_depth')
plt.ylabel('Accuracy')
plt.title('Titanic: Tree Depth vs Accuracy (Bias-Variance Tradeoff)')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\n✅ Best depth by CV: {best_d}")
print(f"   At depth {best_d}: Train={train_acc[best_d-1]:.3f} | Test={test_acc[best_d-1]:.3f} | CV={cv_acc[best_d-1]:.3f}")

---
## Part C: Hyperparameter Tuning (GridSearchCV)

### 🎯 Kya Seekhenge?
> Sirf `max_depth` nahi, decision tree mein **multiple hyperparameters** hote hain. GridSearchCV automatically sabhi combinations try karta hai aur best dhundh leta hai.

```python
param_grid = {
    'max_depth': [3, 5, 7, 10, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 5],
    'criterion': ['gini', 'entropy']
}
# Yeh = 5 × 3 × 3 × 2 = 90 combinations!
# Har combination pe 5-fold CV = 450 models fit honge
```

In [ ]:
# ===========================================
# GridSearchCV: Best Hyperparameters Dhundho
# ===========================================
param_grid = {
    'max_depth':         [3, 5, 7, 10, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf':  [1, 2, 5],
    'criterion':         ['gini', 'entropy']
}

total_combos = 1
for v in param_grid.values(): total_combos *= len(v)
print(f"Total combinations: {total_combos}")
print(f"With 5-fold CV: {total_combos * 5} models fit honge")
print("Running...")

grid_search = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=0
)
grid_search.fit(X_train, y_train)

print(f"\n✅ Best Parameters: {grid_search.best_params_}")
print(f"Best CV Score: {grid_search.best_score_:.4f}")

# Best model evaluate karo
best_dt = grid_search.best_estimator_
y_pred_best = best_dt.predict(X_test)
print(f"\nTest Accuracy (best model): {accuracy_score(y_test, y_pred_best):.4f}")
print(f"Tree Depth: {best_dt.get_depth()} | Leaves: {best_dt.get_n_leaves()}")

In [ ]:
# GridSearch Results: depth aur criterion comparison
results_df = pd.DataFrame(grid_search.cv_results_)

# Pivot: depth vs criterion (avg over other params)
pivot = results_df.pivot_table(
    values='mean_test_score',
    index='param_max_depth',
    columns='param_criterion',
    aggfunc='max'
)

plt.figure(figsize=(8, 4))
sns.heatmap(pivot, annot=True, fmt='.3f', cmap='YlOrRd',
            linewidths=0.5, annot_kws={'size': 10})
plt.title('GridSearchCV: Max CV Accuracy\n(max_depth × criterion)')
plt.ylabel('max_depth')
plt.tight_layout()
plt.show()

In [ ]:
# Best tree visualize karo (max depth 3 tak show karenge readability ke liye)
plt.figure(figsize=(20, 8))
plot_tree(best_dt,
          feature_names=feature_cols,
          class_names=['Died', 'Survived'],
          filled=True,
          rounded=True,
          max_depth=3,  # Display 3 levels only
          fontsize=9,
          impurity=True)
plt.title('Best Decision Tree (Top 3 Levels) — Titanic\nBlue=Survived | Orange=Died', fontsize=12)
plt.tight_layout()
plt.show()

print("\n📋 Top 3 levels ka text representation:")
print(export_text(best_dt, feature_names=feature_cols, max_depth=3))

---
## Part D: Model Interpretation + Error Analysis

### 🎯 Kya Seekhenge?
> Model ne kya seekha? Kaunse features most important hain? Kahan galti ho rahi hai?
>
> Yeh part **explain karna sikhata hai** ki model kaise decide karta hai — jo real projects mein bahut zaroori hai. Doctor ya bank manager ko sirf accuracy nahi chahiye — woh samajhna chahte hain 'kyun?'

**Feature Importance:**
> Decision Tree mein feature importance = uss feature ne total Gini reduction mein kitna contribute kiya

In [ ]:
# ===========================================
# Feature Importance Analysis
# ===========================================
importances = pd.Series(best_dt.feature_importances_, index=feature_cols).sort_values(ascending=True)

colors = ['#e74c3c' if v > 0.15 else ('#f39c12' if v > 0.05 else '#3498db') for v in importances]

plt.figure(figsize=(8, 4))
importances.plot(kind='barh', color=colors, edgecolor='black', alpha=0.85)
for i, (name, val) in enumerate(importances.items()):
    plt.text(val + 0.001, i, f'{val:.3f}', va='center', fontsize=9)
plt.xlabel('Importance Score')
plt.title('Feature Importance — Titanic Decision Tree\n(Red = High, Orange = Medium, Blue = Low)')
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

print("\n📊 Feature Importance:")
print(importances.sort_values(ascending=False).to_string())
print("\n📌 sex_enc (female=1) sabse important hai — 'women first' rule capture ho gaya!")

In [ ]:
# ===========================================
# Error Analysis: Galat Predictions
# ===========================================
X_test_reset = X_test.reset_index(drop=True)
y_test_reset = y_test.reset_index(drop=True)

error_df = X_test_reset.copy()
error_df['Actual']    = y_test_reset.values
error_df['Predicted'] = y_pred_best
error_df['Correct']   = (error_df['Actual'] == error_df['Predicted'])
error_df['sex_label'] = error_df['sex_enc'].map({1: 'Female', 0: 'Male'})

print(f"Total Test: {len(error_df)}")
print(f"Correct:    {error_df['Correct'].sum()} ({error_df['Correct'].mean():.1%})")
print(f"Wrong:      {(~error_df['Correct']).sum()} ({(~error_df['Correct']).mean():.1%})")

# Error types
wrong = error_df[~error_df['Correct']]
fn = wrong[wrong['Actual']==1]  # False Negative: Survived tha, model ne Died predict kiya
fp = wrong[wrong['Actual']==0]  # False Positive: Died tha, model ne Survived predict kiya

print(f"\nFalse Negatives (Missed Survivors): {len(fn)}")
print(f"False Positives (Wrong Survivors):  {len(fp)}")

print("\n📋 Detailed Report:")
print(classification_report(y_test, y_pred_best, target_names=['Died', 'Survived']))

In [ ]:
# Error patterns visualize karo
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# 1. Error by Gender
err_sex = error_df.groupby('sex_label')['Correct'].mean()
err_sex.plot(kind='bar', ax=axes[0], color=['#e74c3c','#3498db'], rot=0)
axes[0].set_title('Accuracy by Gender')
axes[0].set_ylabel('Accuracy')
axes[0].set_ylim(0, 1)
for i, v in enumerate(err_sex):
    axes[0].text(i, v+0.01, f'{v:.1%}', ha='center', fontweight='bold')

# 2. Error by Pclass
err_class = error_df.groupby('pclass')['Correct'].mean()
err_class.plot(kind='bar', ax=axes[1], color=['gold','silver','#cd7f32'], rot=0)
axes[1].set_title('Accuracy by Passenger Class')
axes[1].set_xlabel('Pclass')
axes[1].set_ylim(0, 1)
for i, v in enumerate(err_class):
    axes[1].text(i, v+0.01, f'{v:.1%}', ha='center', fontweight='bold')

# 3. Age distribution: Correct vs Wrong
error_df[error_df['Correct']]['age'].plot(kind='hist', ax=axes[2], alpha=0.6,
                                           label='Correct', color='#2ecc71', bins=15)
error_df[~error_df['Correct']]['age'].plot(kind='hist', ax=axes[2], alpha=0.6,
                                            label='Wrong', color='#e74c3c', bins=15)
axes[2].set_title('Age: Correct vs Wrong Predictions')
axes[2].legend()

plt.suptitle('Error Analysis — Where Does the Model Fail?', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ===========================================
# Single Passenger ka Prediction Explain Karo
# ===========================================
# Ek passenger le lo test set se
passenger_idx = 5
passenger = X_test.iloc[passenger_idx]
actual = y_test.iloc[passenger_idx]
pred = best_dt.predict([passenger])[0]
prob = best_dt.predict_proba([passenger])[0]

print("=" * 50)
print("🧍 Passenger Details:")
print(f"  pclass     : {passenger['pclass']} ({'1st' if passenger['pclass']==1 else '2nd' if passenger['pclass']==2 else '3rd'} Class)")
print(f"  sex        : {'Female' if passenger['sex_enc']==1 else 'Male'}")
print(f"  age        : {passenger['age']:.1f} years")
print(f"  fare       : ${passenger['fare']:.2f}")
print(f"  family_size: {passenger['family_size']}")
print("=" * 50)
print(f"  Actual:    {'Survived ✅' if actual==1 else 'Died ❌'}")
print(f"  Predicted: {'Survived ✅' if pred==1 else 'Died ❌'}")
print(f"  P(Died)={prob[0]:.3f} | P(Survived)={prob[1]:.3f}")
print("=" * 50)

In [ ]:
# ===========================================
# Final Model Comparison Summary
# ===========================================
models = {
    'Default Tree (no limit)': DecisionTreeClassifier(random_state=42),
    'Shallow Tree (depth=3)':  DecisionTreeClassifier(max_depth=3, random_state=42),
    'Best (GridSearchCV)':     best_dt
}

summary = []
for name, model in models.items():
    if name != 'Best (GridSearchCV)':
        model.fit(X_train, y_train)
    cv = cross_val_score(model, X, y, cv=5)
    train_a = model.score(X_train, y_train)
    test_a  = model.score(X_test, y_test)
    summary.append({
        'Model': name,
        'Depth': model.get_depth(),
        'Train Acc': round(train_a, 4),
        'Test Acc':  round(test_a, 4),
        'CV Mean':   round(cv.mean(), 4),
        'CV Std':    round(cv.std(), 4)
    })

df_summary = pd.DataFrame(summary)
print("\n=== Final Model Comparison ===")
print(df_summary.to_string(index=False))

print("\n📌 Conclusion:")
print("  GridSearchCV se mila best model: best balance of train/test/cv scores")
print("  Default tree clearly overfit hai (Train=1.0, lekin test kam)")

---
## 🎯 Is Notebook Ka Summary — Humne Kya Seekha?

| Step | Kya Kiya | Kya Seekha |
|------|----------|------------|
| EDA | Survival patterns dekhe | Sex aur Pclass survival ke strongest predictors hain |
| Preprocessing | Missing values fill, encoding | DT ko numeric input chahiye |
| Baseline | Default tree fit kiya | Unlimited depth = overfit |
| Depth Analysis | Depth vary kiya | Sweet spot dhundha via CV |
| GridSearchCV | 90 combos try kiya | Best hyperparams auto-dhunde |
| Feature Importance | Importance plot | sex_enc (gender) most important |
| Error Analysis | Galat predictions analyze ki | Pclass 2 mein zyada errors |

### 💡 Real World Mein Kaise Use Karein?

> **HR Analytics:** Employee resign karega? → Features: tenure, salary, performance
> 
> **Healthcare:** Patient admitted hoga emergency mein? → Features: age, vitals, history
>
> **Banking:** Fraud transaction hai? → Features: amount, location, time, merchant
>
> Decision Tree pehle ek **interpretable baseline** deta hai. Agar accuracy enough nahi toh Random Forest ya XGBoost try karo!

---
## 🏋️ Capstone: Part A-D — Tumhari Baari!

**Dataset:** Iris (Multi-class classification)
```python
from sklearn.datasets import load_iris
iris = load_iris()
```
**Task:** Iris flower ka species predict karo (3 classes)

In [ ]:
from sklearn.datasets import load_iris
iris = load_iris()
X_iris = pd.DataFrame(iris.data, columns=iris.feature_names)
y_iris = iris.target

print("Iris Dataset:")
print(f"  Samples: {X_iris.shape[0]}")
print(f"  Features: {list(iris.feature_names)}")
print(f"  Classes: {list(iris.target_names)}")

X_tr, X_te, y_tr, y_te = train_test_split(X_iris, y_iris, test_size=0.2, random_state=42, stratify=y_iris)

In [ ]:
# PART A: EDA
# TODO 1: Pairplot ya scatter matrix banao — kaunse features classes separate karte hain?
# YOUR CODE HERE

print("Part A complete karo!")

In [ ]:
# PART B: Model + Depth
# TODO 2: Depth 1 se 10 tak tree fit karo, train/test/cv accuracy plot karo
# YOUR CODE HERE

print("Part B complete karo!")

In [ ]:
# PART C: GridSearchCV
# TODO 3: GridSearchCV se best hyperparams dhundo
# param_grid = {'max_depth': [2,3,5,None], 'min_samples_leaf': [1,2,5], 'criterion': ['gini','entropy']}
# YOUR CODE HERE

print("Part C complete karo!")

In [ ]:
# PART D: Visualization + Interpretation
# TODO 4: Best model ka tree visualize karo (plot_tree)
# TODO 5: Feature importance bar chart banao
# TODO 6: Iris mein kaunse features most important nikle? Kya expected tha EDA se?
# YOUR CODE HERE

print("Part D complete karo!")